# Module: From Point Estimates to Statistical Uncertainty

Statistics exists because data is incomplete. If we could measure every individual in a population, there would be no uncertainty. We would simply calculate the true population mean:

$$\mu = \frac{\sum X}{N}$$

where:
- $\mu$ = population mean
- $N$ = population size
- $X$ = each population observation

But in real-world problems, complete population data is usually impossible or impractical to obtain:
- surveying every voter
- testing every manufactured chip
- measuring every patient's blood pressure
- checking every transaction for fraud

Instead, we collect a sample and calculate the sample mean:

$$\bar{x} = \frac{\sum x_i}{n}$$

where $\bar{x}$ is the sample mean, $n$ is the sample size, and $x_i$ are individual sample observations. The sample mean acts as an estimator of the population mean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Simulating Population vs Sample
np.random.seed(42)
population = np.random.normal(loc=50, scale=10, size=10000)
true_mu = np.mean(population)

sample_size = 100
sample = np.random.choice(population, size=sample_size, replace=False)
sample_mean = np.mean(sample)

print(f'True Population Mean (μ): {true_mu:.2f}')
print(f'Sample Mean (x̄) for n={sample_size}: {sample_mean:.2f}')

## 2. Why Point Estimates Are Fundamentally Incomplete

Suppose two analysts independently sample student marks from the same university.
- Analyst A gets: $\bar{x} = 72$
- Analyst B gets: $\bar{x} = 76$

Neither analyst is necessarily wrong. This happens because of **sampling variability**. Different samples produce different statistics. 

> **Key Insight:** A statistic is itself a random variable.

In [ ]:
# Demonstrating Sampling Variability
analyst_a_sample = np.random.choice(population, size=50)
analyst_b_sample = np.random.choice(population, size=50)

print(f'Analyst A Sample Mean: {np.mean(analyst_a_sample):.2f}')
print(f'Analyst B Sample Mean: {np.mean(analyst_b_sample):.2f}')
print('Both are valid estimates of the same population!')

## 3. Sampling Distribution of the Mean

If we repeatedly draw samples of size $n$ and compute their means ($\bar{x}_1, \bar{x}_2, \bar{x}_3, \dots$), these means form a distribution.

- **Mean of the Sampling Distribution:** $E(\bar{x}) = \mu$ (The sample mean is an unbiased estimator).
- **Standard Deviation of the Sampling Distribution:** $\sigma_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$

This quantity is called the **standard error of the mean**. To cut uncertainty in half, you must quadruple the sample size.

In [ ]:
# Simulating the Sampling Distribution
num_samples = 1000
sample_means = [np.mean(np.random.choice(population, size=30)) for _ in range(num_samples)]

plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=30, density=True, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(true_mu, color='red', linestyle='dashed', linewidth=2, label=f'True μ ({true_mu:.2f})')
plt.title('Sampling Distribution of the Mean (n=30)')
plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.legend()
plt.show()

## 4. Central Limit Theorem (CLT)

The CLT states: *For sufficiently large sample sizes, the sampling distribution of the sample mean becomes approximately normal, regardless of the population distribution.*

Formally: $\bar{x} \sim N\left(\mu, \frac{\sigma}{\sqrt{n}}\right)$ for large $n$.

This is astonishingly powerful because populations can be skewed, heavy-tailed, or irregular, and yet sample means behave predictably.

In [ ]:
# CLT Demonstration: Highly Skewed Population
skewed_pop = np.random.exponential(scale=2.0, size=10000)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(skewed_pop, bins=50, density=True, color='salmon')
plt.title('Original Population (Exponential/Skewed)')

plt.subplot(1, 2, 2)
clt_means = [np.mean(np.random.choice(skewed_pop, size=100)) for _ in range(1000)]
plt.hist(clt_means, bins=30, density=True, color='lightgreen', edgecolor='black')
plt.title('Sampling Distribution of Mean (n=100)')
plt.show()

## 5. Confidence Intervals

A confidence interval converts uncertainty into a quantitative range.

$$\text{Confidence Interval} = \text{Point Estimate} \pm \text{Margin of Error}$$

Equivalently:
$$\text{CI} = \text{Estimator} \pm (\text{Critical Value}) \times (\text{Standard Error})$$

## 6. Understanding the Margin of Error

$$E = (\text{Critical Value}) \times (\text{Standard Error})$$

The margin of error grows when variability increases, confidence level increases, or sample size decreases. This creates the core statistical tradeoff:

| Goal | Effect |
|---|---|
| Higher confidence | Wider interval |
| More precision | Narrower interval |
| Small sample | Less precision |
| Large sample | More precision |

In [ ]:
# Visualizing the Margin of Error Tradeoff
sample_sizes = [10, 30, 100, 500]
sigma = 15
z_crit = 1.96

margins_of_error = [z_crit * (sigma / np.sqrt(n)) for n in sample_sizes]

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, margins_of_error, marker='o', color='purple', linewidth=2)
plt.title('Margin of Error vs. Sample Size')
plt.xlabel('Sample Size (n)')
plt.ylabel('Margin of Error (E)')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Confidence Level and Significance Level

Suppose confidence level is 95%. Then $\alpha = 1 - 0.95 = 0.05$.
- $\alpha$ = significance level (total tail probability = 0.05)
- For a two-sided interval: $\frac{\alpha}{2} = 0.025$ (2.5% area in each tail)
- For the standard normal distribution: $Z_{0.025} = 1.96$

In [ ]:
# Visualizing Alpha and Critical Values
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(8, 5))
plt.plot(x, y, 'b-', lw=2)
plt.fill_between(x[x <= -1.96], y[x <= -1.96], color='red', alpha=0.3, label='α/2 = 0.025')
plt.fill_between(x[x >= 1.96], y[x >= 1.96], color='red', alpha=0.3)
plt.title('Standard Normal Distribution: 95% Confidence (α = 0.05)')
plt.axvline(-1.96, color='red', linestyle='--')
plt.axvline(1.96, color='red', linestyle='--')
plt.legend()
plt.show()

## 8. Z-Interval for the Mean

When population standard deviation $\sigma$ is known:
$$\text{CI} = \bar{x} \pm Z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$$

**Conditions:** random sample, independent observations, population normal OR large sample size, $\sigma$ known.

In [ ]:
# Example of a Z-Interval
x_bar = 75
sigma = 12
n = 64
confidence = 0.95
z_critical = stats.norm.ppf((1 + confidence) / 2)

se = sigma / np.sqrt(n)
margin_of_error = z_critical * se
ci_lower = x_bar - margin_of_error
ci_upper = x_bar + margin_of_error

print(f'Standard Error: {se}')
print(f'Critical Value (Z): {z_critical}')
print(f'Margin of Error: {margin_of_error:.2f}')
print(f'95% Confidence Interval: ({ci_lower:.2f}, {ci_upper:.2f})')

## 9. Why the Z-Interval Is Rare in Practice

In real applications, population standard deviation $\sigma$ is almost never known. Replacing $\sigma$ with sample standard deviation $s$ fundamentally changes the uncertainty structure. Additional variability enters the problem, which must be modeled using the **Student's t-distribution**.

## 10. The Student's t-Distribution & Degrees of Freedom

Developed by William Sealy Gosset (pseudonym "Student") at Guinness. Compared to the normal distribution, the t-distribution is:
- symmetric, bell-shaped, centered at zero
- **heavier tails**, more spread, more conservative

Why heavier tails? Because estimating $\sigma$ introduces uncertainty. The t-distribution "admits ignorance" more honestly.

**Degrees of Freedom:** $df = n - 1$. This represents how much independent information remains after estimating parameters.

In [ ]:
# Plotting Normal vs t-Distribution
x = np.linspace(-4, 4, 1000)
plt.figure(figsize=(8, 5))
plt.plot(x, stats.norm.pdf(x), label='Standard Normal (Z)', lw=2, color='blue')
plt.plot(x, stats.t.pdf(x, df=4), label='t-distribution (df=4)', lw=2, color='red', linestyle='--')
plt.plot(x, stats.t.pdf(x, df=15), label='t-distribution (df=15)', lw=2, color='green', linestyle='-.')
plt.title('t-Distribution vs Normal Distribution')
plt.xlabel('x')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. T-Interval for the Mean

When $\sigma$ is unknown:
$$\text{CI} = \bar{x} \pm t_{\alpha/2, df} \frac{s}{\sqrt{n}}$$

This is the most common confidence interval used in introductory statistics.

In [ ]:
# Example of a T-Interval
x_bar_t = 82
s = 10
n_t = 25
df = n_t - 1
t_critical = stats.t.ppf((1 + 0.95) / 2, df=df)

se_t = s / np.sqrt(n_t)
me_t = t_critical * se_t

print(f'Degrees of Freedom: {df}')
print(f'Critical Value (t): {t_critical:.3f}')
print(f'Standard Error: {se_t}')
print(f'Margin of Error: {me_t:.3f}')
print(f'95% Confidence Interval: ({x_bar_t - me_t:.3f}, {x_bar_t + me_t:.3f})')

## 12. Why Small Samples Are Dangerous

Small samples create multiple problems simultaneously:
- unstable sample means
- unstable standard deviations
- sensitivity to outliers
- non-normality effects amplified

The t-distribution compensates partially through heavier tails, but it does not magically fix poor sampling. Statistics cannot rescue fundamentally bad data collection.

In [ ]:
# Demonstrating Small Sample Instability
true_mean = 50
small_sample_means = [np.mean(np.random.normal(true_mean, 10, size=5)) for _ in range(1000)]
large_sample_means = [np.mean(np.random.normal(true_mean, 10, size=100)) for _ in range(1000)]

plt.figure(figsize=(10, 5))
plt.hist(small_sample_means, bins=40, density=True, alpha=0.6, label='n=5 (High Variance)')
plt.hist(large_sample_means, bins=40, density=True, alpha=0.6, label='n=100 (Low Variance)')
plt.axvline(true_mean, color='black', linestyle='dashed', label='True Mean')
plt.title('Effect of Sample Size on Estimator Stability')
plt.legend()
plt.show()

## 13. Factors Affecting Interval Width

1. **Sample Size ($n$):** Larger $n \rightarrow$ smaller standard error $\rightarrow$ narrower interval.
2. **Variability ($\sigma$ or $s$):** Larger variability $\rightarrow$ larger standard error $\rightarrow$ wider interval.
3. **Confidence Level:** Higher confidence $\rightarrow$ larger critical value $\rightarrow$ wider interval.

| Confidence Level | Z Critical Value |
|---|---|
| 90% | 1.645 |
| 95% | 1.960 |
| 99% | 2.576 |

In [ ]:
# Visualizing Confidence Level vs Interval Width
conf_levels = [0.90, 0.95, 0.99]
z_values = [stats.norm.ppf((1 + cl) / 2) for cl in conf_levels]
widths = [2 * z * (10 / np.sqrt(30)) for z in z_values] # Assuming s=10, n=30

plt.figure(figsize=(8, 5))
plt.bar([f'{int(cl*100)}%' for cl in conf_levels], widths, color=['lightblue', 'skyblue', 'royalblue'])
plt.title('Confidence Interval Width vs Confidence Level')
plt.ylabel('Interval Width')
plt.show()

## 14. One-Sided Confidence Intervals

Not all inference problems are symmetric. Sometimes we only care about lower bounds (e.g., minimum battery life) or upper bounds (e.g., maximum defect rate).

- Lower confidence bound: $\bar{x} - t_{\alpha, df}\frac{s}{\sqrt{n}}$
- Upper confidence bound: $\bar{x} + t_{\alpha, df}\frac{s}{\sqrt{n}}$

One-sided intervals place all significance level $\alpha$ into one tail.

In [ ]:
# One-Sided Confidence Interval Example
# We want to be 95% confident the mean is AT LEAST some value (Lower Bound)
alpha_one_sided = 0.05
t_crit_one_sided = stats.t.ppf(1 - alpha_one_sided, df=24)
lower_bound = x_bar_t - (t_crit_one_sided * se_t)

print(f'One-Sided Critical Value (t, α=0.05, df=24): {t_crit_one_sided:.3f}')
print(f'We are 95% confident the true mean is greater than: {lower_bound:.3f}')

## 15. Confidence Intervals vs Hypothesis Testing

These are deeply connected. For a two-sided hypothesis test:
- If the hypothesized parameter value lies **outside** the confidence interval: $H_0$ is rejected.
- If it lies **inside**: $H_0$ is not rejected.

Confidence intervals often provide more information because they show effect size, uncertainty, and plausible parameter range, rather than merely "reject/not reject."

In [ ]:
# Linking CI to Hypothesis Testing
null_hypothesis_mean = 80
ci_lower, ci_upper = x_bar_t - me_t, x_bar_t + me_t

print(f'95% CI: ({ci_lower:.3f}, {ci_upper:.3f})')
print(f'Null Hypothesis (μ0): {null_hypothesis_mean}')

if null_hypothesis_mean < ci_lower or null_hypothesis_mean > ci_upper:
    print('Conclusion: Reject H0 (The null value is outside the CI)')
else:
    print('Conclusion: Fail to reject H0 (The null value is inside the CI)')

## 16. Confidence Intervals as Signal vs Noise

The estimate $\bar{x}$ is the **signal**. Sampling variability is the **noise**. The interval quantifies how uncertain we are about separating the two.

In practice, confidence intervals matter because decision-making under uncertainty matters in medicine, finance, manufacturing, machine learning, public policy, and forecasting. **The interval is often more important than the estimate itself.** A point estimate without uncertainty is statistically incomplete.

## 17. Conclusions & Summary

Confidence intervals move us from a single "best guess" to a range of plausible values.

### Anatomy of a Confidence Interval
$$\text{CI} = \text{Point Estimate} \pm (\text{Critical Value} \times \text{Standard Error})$$

### Choosing the Correct Distribution
| Scenario | Known $\sigma$ | Unknown $\sigma$ |
|---|---|---|
| **Distribution** | Standard Normal ($Z$) | Student's $t$ |
| **Formula** | $\bar{x} \pm Z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$ | $\bar{x} \pm t_{\alpha/2, df} \frac{s}{\sqrt{n}}$ |

> **The Correct Interpretation:** We are 95% confident in the **method**. If we repeated this sampling process many times, we expect 95% of the resulting intervals to successfully capture the true population mean ($\mu$).

In [ ]:
# Utility Function: Calculate and Print Confidence Interval
def calculate_ci(data, confidence=0.95, sigma_known=None):
    n = len(data)
    x_bar = np.mean(data)
    
    if sigma_known is not None:
        se = sigma_known / np.sqrt(n)
        crit_val = stats.norm.ppf((1 + confidence) / 2)
        dist_name = 'Z (Normal)'
    else:
        s = np.std(data, ddof=1)
        se = s / np.sqrt(n)
        crit_val = stats.t.ppf((1 + confidence) / 2, df=n-1)
        dist_name = f"t (df={n-1})"
        
    me = crit_val * se
    print(f'--- {int(confidence*100)}% Confidence Interval ({dist_name}) ---')
    print(f'Sample Mean: {x_bar:.2f}')
    print(f'Standard Error: {se:.2f}')
    print(f'Critical Value: {crit_val:.3f}')
    print(f'Margin of Error: {me:.2f}')
    print(f'Interval: ({x_bar - me:.2f}, {x_bar + me:.2f})\n')
    return (x_bar - me, x_bar + me)

# Test the function
sample_data = [82, 85, 79, 88, 81, 84, 80, 83, 86, 78]
calculate_ci(sample_data, confidence=0.95)